<a href="https://colab.research.google.com/github/szymonbloch/tuberculosis_detection/blob/main/preparing_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import glob

COL_LABEL = 0
COL_DECISION = 2
COL_EXCLUDE = 5

LABELS_POS_NEG_PATH = '/content/drive/MyDrive/NTWI_project/data/Etykiety_1-layer-slides.xlsx'

df = pd.read_excel(LABELS_POS_NEG_PATH)
df.iloc[:, COL_DECISION] = df.iloc[:, COL_DECISION].astype(str).str.strip().str.lower()
df_one_layer_label = df[(df.iloc[:, COL_DECISION].isin(['pos', 'neg'])) &
                (df.iloc[:, COL_EXCLUDE].isnull())]

df_final = df_one_layer_label.iloc[:, [COL_LABEL, COL_DECISION]].copy()
df_final.columns = ['image_name', 'output_image']
df_final['output_image'] = df_final['output_image'].map({'neg': 0, 'pos': 1})
df_final['image_name'] = df_final['image_name'].str.split('.').str[0]


In [ ]:
print(df_final.head())
len(df_final)

  image_name  output_image
0    AFB-001             1
1    AFB-002             1
2    AFB-003             1
3    AFB-004             1
5    AFB-006             0


136

In [ ]:
ANOMALY_DIRECTORY_PATH = '/content/drive/MyDrive/NTWI_project/data/anomaly_detector/1-layer_ad_cnn_04_06_2025_00_31_21/*.csv'

csv_files = glob.glob(ANOMALY_DIRECTORY_PATH)
table_list = []

for file in csv_files:
  df_temp = pd.read_csv(file)
  df_temp.rename(columns={'image_name': 'tile_name', 'output': 'tile_output'}, inplace=True)

  df_temp['probability'] = df_temp['probability'].astype(str).str.extract(r'tensor\(([0-9.]+)').astype(float)
  df_temp['tile_name'] = df_temp['tile_name'].str.split('.').str[0].str.strip()
  df_temp['image_name'] = df_temp['tile_name'].str.split('_').str[0].str.strip()

  table_list.append(df_temp)

df_anomaly = pd.concat(table_list, ignore_index=True)


In [ ]:
len(table_list)
print(df_anomaly.head())
len(df_anomaly)

               tile_name  tile_output  probability image_name
0  AFB-001_0_56832_33280            0       0.0007    AFB-001
1   AFB-001_0_5632_46592            0       0.0008    AFB-001
2   AFB-001_0_8960_36864            0       0.0008    AFB-001
3  AFB-001_0_41472_69888            0       0.0008    AFB-001
4  AFB-001_0_37888_47104            0       0.0007    AFB-001


4528219

In [ ]:
COLORS_DIRECTORY_PATH = '/content/drive/MyDrive/NTWI_project/data/color_features/1_layer/*.csv'

csv_files = glob.glob(COLORS_DIRECTORY_PATH)
table_list = []

for file in csv_files:
  df_temp = pd.read_csv(file, header=None, sep=None, engine='python')
  df_temp.rename(columns={0: 'tile_name'}, inplace=True)
  df_temp['tile_name'] = df_temp['tile_name'].str.split('.').str[0].str.strip()

  table_list.append(df_temp)

df_colors = pd.concat(table_list, ignore_index=True)

In [ ]:
len(df_colors)
print(df_colors.head())

                tile_name   1    2         3      4         5      6  \
0  AFB-136_0_10112_114432  82  252  14523061  65536  14850938  65536   
1  AFB-136_0_10112_115456  58  254  13782286  65536  14423841  65536   
2   AFB-136_0_10112_78592  54  253  14465245  65536  14948481  65536   
3   AFB-136_0_10112_78848  41  251  13409549  65536  14256694  65536   
4   AFB-136_0_10112_79104  47  250  12911066  65536  13974468  65536   

          7      8     9  ...  50  51  52  53  54  55  56  57  58  59  
0  15199481  65536  5519  ...   0   0   0   0   0   0   0   0   0 NaN  
1  15053318  65536  3753  ...   0   0   0   0   0   0   0   0   0 NaN  
2  15313635  65536  1691  ...   0   0   0   0   0   0   0   0   0 NaN  
3  14945436  65536   488  ...   0   0   0   0   0   0   0   0   0 NaN  
4  14912769  65536   231  ...   0   0   0   0   0   0   0   0   0 NaN  

[5 rows x 60 columns]


Approach with statistics and random forest

1. Data merging

In [ ]:
df_merged = df_anomaly.merge(df_final, on='image_name', how='inner').merge(df_colors, on='tile_name', how='inner')
df_merged = df_merged.dropna(axis=1, how='all') # Usunięcie ostatniej pustej kolumny

In [ ]:
print("Liczba wierszy (kafli):", len(df_merged))
print("Liczba kolumn (cech):", len(df_merged.columns))
print(df_merged.head())

Liczba wierszy (kafli): 3155535
Liczba kolumn (cech): 63
               tile_name  tile_output  probability image_name  output_image  \
0  AFB-001_0_56832_33280            0       0.0007    AFB-001             1   
1   AFB-001_0_5632_46592            0       0.0008    AFB-001             1   
2   AFB-001_0_8960_36864            0       0.0008    AFB-001             1   
3  AFB-001_0_41472_69888            0       0.0008    AFB-001             1   
4  AFB-001_0_37888_47104            0       0.0007    AFB-001             1   

     1    2         3      4         5  ...  49  50  51  52  53  54  55  56  \
0   93  255  15076976  65536  14964763  ...   0   0   0   0   0   0   0   0   
1   93  255  15059846  65536  14919858  ...   0   0   0   0   0   0   0   0   
2   77  255  15069769  65536  14935905  ...   0   0   0   0   0   0   0   0   
3  131  255  15171065  65536  14931638  ...   0   0   0   0   0   0   0   0   
4   52  255  14992676  65536  14798151  ...   0   0   0   0   0   0   0  

In [ ]:
df_merged.to_csv('/content/drive/MyDrive/NTWI_project/results/df_merged.csv')

In [ ]:
MERGED_DATAFRAME_PATH = '/content/drive/MyDrive/NTWI_project/results/df_merged.csv'

df_merged = pd.read_csv(LABELS_POS_NEG_PATH)

statistic_dict = {
    'probability': ['mean', 'max', 'std'],
    'output_image': ['first']
}

for col in df_merged.columns:
    if type(col) == int or str(col).isdigit():
        statistic_dict[col] = ['mean']
df_statistics = df_merged.groupby('image_name').agg(statistic_dict).reset_index()
df_statistics.columns = ['_'.join([str(c) for c in col]).strip('_') for col in df_statistics.columns.values]
df_statistics.rename(columns={'output_image_first': 'output_image'}, inplace=True)

In [ ]:
print("Liczba slajdów (pacjentów):", len(df_statistics))
print("Liczba statystycznych cech:", len(df_statistics.columns))
print(df_statistics.head())

Liczba slajdów (pacjentów): 133
Liczba statystycznych cech: 63
  image_name  probability_mean  probability_max  probability_std  \
0    AFB-001          0.000828           0.0046         0.000193   
1    AFB-002          0.001145           0.0038         0.000278   
2    AFB-003          0.000870           0.0018         0.000172   
3    AFB-004          0.001611           0.0039         0.000566   
4    AFB-006          0.000744           0.0016         0.000231   

   output_image      1_mean      2_mean        3_mean        4_mean  \
0             1   96.281867  254.673199  1.467378e+07  64700.453633   
1             1   81.000502  254.821320  1.401996e+07  65389.330018   
2             1  102.205684  254.836891  1.461163e+07  64883.599532   
3             1   37.353746  248.508900  1.138322e+07  65088.762927   
4             0   50.412028  245.569416  1.109618e+07  64604.426957   

         5_mean  ...  49_mean  50_mean  51_mean  52_mean  53_mean  54_mean  \
0  1.447040e+07  ...   

In [ ]:
df.to_csv('/content/drive/MyDrive/NTWI_project/results/df_statistics.csv')

ML model training (Random Forest)


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import balanced_accuracy_score, confusion_matrix

STATISTICS_DATAFRAME_PATH = '/content/drive/MyDrive/NTWI_project/results/df_statistics.csv'

df_statistics = pd.read_csv(STATISTICS_DATAFRAME_PATH)

# 2. Podział na cechy (X) i ostateczną diagnozę (y)
X = df_statistics.drop(columns=['image_name', 'output_image'])
y = df_statistics['output_image']

# 3. Odłożenie 20% pacjentów do testowania modelu
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 4. Budowa i trening Głównego Lekarza (Random Forest)
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
rf_model.fit(X_train, y_train)

# 5. Egzamin na ukrytych danych
y_pred = rf_model.predict(X_test)
bacc = balanced_accuracy_score(y_test, y_pred)


KeyError: "['image_name', 'output_image'] not found in axis"

In [ ]:
print(f"\nWynik Balanced Accuracy: {bacc * 100:.2f}%")
print("Macierz pomyłek (Prawdziwe vs Przewidziane):\n", confusion_matrix(y_test, y_pred))


Wynik Balanced Accuracy: 79.12%
Macierz pomyłek (Prawdziwe vs Przewidziane):
 [[15  2]
 [ 3  7]]


Approach 2 (Multiple Instance Learning)
- Multiple instance learning na całych danych mija się z celem ZA DUŻO RAMU POŻERA

In [ ]:
import pandas as pd
import numpy as np

MERGED_DATAFRAME_PATH = '/content/drive/MyDrive/NTWI_project/results/df_merged.csv'

df_merged = pd.read_csv(MERGED_DATAFRAME_PATH)

bags = []
labels = []

# Grupujemy dane po pacjencie
grouped = df_merged.groupby('image_name')

for name, group in grouped:
    # Pobieramy ostateczną decyzję dla tego pacjenta (1 lub 0)
    label = group['output_image'].iloc[0]

    # Wycinamy tylko kolumny z cechami (prawdopodobieństwo + kolory)
    # Zostawiamy 'probability' oraz wszystkie kolumny numeryczne (kolory od 1 do 58)
    features_cols = ['probability'] + [col for col in group.columns if str(col).isdigit()]
    features = group[features_cols].values

    bags.append(features)
    labels.append(label)

labels = np.array(labels)

print(f"Liczba przygotowanych worków (pacjentów): {len(bags)}")
print(f"Kształt pierwszego worka: {bags[0].shape} (Kafelki x Cechy)")

Liczba przygotowanych worków (pacjentów): 133
Kształt pierwszego worka: (18828, 59) (Kafelki x Cechy)


Biblioteka misvm

In [ ]:
!pip install git+https://github.com/garydoranjr/misvm.git

  Cloning https://github.com/garydoranjr/misvm.git to /tmp/pip-req-build-rw2njw2c
  Running command git clone --filter=blob:none --quiet https://github.com/garydoranjr/misvm.git /tmp/pip-req-build-rw2njw2c
  Resolved https://github.com/garydoranjr/misvm.git to commit b2118fe04d98c00436bdf8a0e4bbfb6082c5751c
  Preparing metadata (setup.py) ... done


In [ ]:
import misvm
from sklearn.model_selection import train_test_split
from sklearn.metrics import balanced_accuracy_score, confusion_matrix
import numpy as np

# MISVM wymaga etykiet w formacie -1 (zdrowy) i 1 (chory), a nie 0 i 1
labels_svm = np.where(labels == 0, -1, 1)

# Podział na zbiór treningowy i testowy (bags i labels_svm)
bags_train, bags_test, y_train, y_test = train_test_split(
    bags, labels_svm, test_size=0.2, random_state=42, stratify=labels_svm
)

# Inicjalizacja modelu SIL (Single-Instance Learning na poziomie worków)
# Możesz też przetestować misvm.MISVM(), ale dla 8500 kafli SIL liczy się szybciej
classifier = misvm.SIL(kernel='linear', C=1.0)

print("Rozpoczęto trening klasycznego MIL (to może chwilę potrwać)...")
classifier.fit(bags_train, y_train)

# Predykcja
y_pred = classifier.predict(bags_test)

# Konwersja wyników z powrotem na 0 i 1 do oceny
y_pred_bin = np.where(np.array(y_pred) < 0, 0, 1)
y_test_bin = np.where(y_test < 0, 0, 1)

print(f"\nWynik Balanced Accuracy (MISVM): {balanced_accuracy_score(y_test_bin, y_pred_bin) * 100:.2f}%")
print("Macierz pomyłek:\n", confusion_matrix(y_test_bin, y_pred_bin))

Rozpoczęto trening klasycznego MIL (to może chwilę potrwać)...


Approach 2 (Multiple Instance Learning) Ale wybiórczo (top-K)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import balanced_accuracy_score, confusion_matrix
import pandas as pd
import numpy as np

MERGED_DATAFRAME_PATH = '/content/drive/MyDrive/NTWI_project/results/df_merged.csv'

df_merged = pd.read_csv(MERGED_DATAFRAME_PATH)

K = 50 # Bierzemy tylko 50 najbardziej "chorych" kafelków z każdego slajdu

top_k_bags = []
labels = []

grouped = df_merged.groupby('image_name')

for name, group in grouped:
    # 1. Sortujemy kafelki pacjenta malejąco po prawdopodobieństwie z pierwszej sieci
    group_sorted = group.sort_values(by='probability', ascending=False)

    # 2. Wycinamy tylko K pierwszych (najbardziej podejrzanych) kafelków
    top_k_tiles = group_sorted.head(K)

    label = top_k_tiles['output_image'].iloc[0]
    labels.append(label)

    # 3. Z tych 50 kafelków robimy jeden długi, płaski wektor cech (zgniatamy je w jeden wiersz)
    features_cols = ['probability'] + [col for col in group.columns if str(col).isdigit()]
    # Spłaszczamy: 50 kafelków * 59 cech = 2950 cech dla pacjenta
    flat_features = top_k_tiles[features_cols].values.flatten()

    # Zabezpieczenie: jeśli jakiś dziwny pacjent ma mniej niż 50 kafelków ogółem, dopełniamy zerami
    if len(flat_features) < K * len(features_cols):
        padding = np.zeros(K * len(features_cols) - len(flat_features))
        flat_features = np.concatenate((flat_features, padding))

    top_k_bags.append(flat_features)

X_top_k = np.array(top_k_bags)
y_top_k = np.array(labels)

In [ ]:
print(f"Kształt nowej tabeli: {X_top_k.shape} (Pacjenci x Płaskie Cechy)")

Kształt nowej tabeli: (133, 2950) (Pacjenci x Płaskie Cechy)


In [ ]:
# --- TRENING KLASYCZNYM RANDOM FOREST ---
X_train, X_test, y_train, y_test = train_test_split(
    X_top_k, y_top_k, test_size=0.2, random_state=42, stratify=y_top_k
)

rf_model_top_k = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
rf_model_top_k.fit(X_train, y_train)

y_pred = rf_model_top_k.predict(X_test)
bacc = balanced_accuracy_score(y_test, y_pred)

In [ ]:
print(f"\nWynik Balanced Accuracy (Top-{K} MIL z Random Forest): {bacc * 100:.2f}%")
print("Macierz pomyłek:\n", confusion_matrix(y_test, y_pred))


Wynik Balanced Accuracy (Top-50 MIL z Random Forest): 78.24%
Macierz pomyłek:
 [[13  4]
 [ 2  8]]


Pytorch Nie ma sensu

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.metrics import balanced_accuracy_score, confusion_matrix
from sklearn.preprocessing import StandardScaler
import numpy as np

# 1. Podział danych
bags_train, bags_test, y_train, y_test = train_test_split(
    bags, labels, test_size=0.2, random_state=42, stratify=labels
)

# ========================================================
# 2. KLUCZOWA NAPRAWA: Skalowanie (Normalizacja) danych
# ========================================================
scaler = StandardScaler()

# Scaler potrzebuje płaskiej tabeli, żeby wyliczyć średnią i odchylenie z danych treningowych.
# Zlepiamy wszystkie worki treningowe w jedną wielką tabelę kafelków.
wszystkie_kafelki_treningowe = np.vstack(bags_train)
scaler.fit(wszystkie_kafelki_treningowe)

# Teraz "prasujemy" każdy worek pacjenta z osobna używając wyuczonych reguł scalera
bags_train_scaled = [scaler.transform(bag) for bag in bags_train]
bags_test_scaled = [scaler.transform(bag) for bag in bags_test]


# 3. Definicja modelu (Zostaje taka sama)
class AttentionMIL(nn.Module):
    def __init__(self, input_dim):
        super(AttentionMIL, self).__init__()
        self.feature_extractor = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU()
        )
        self.attention = nn.Sequential(
            nn.Linear(32, 16),
            nn.Tanh(),
            nn.Linear(16, 1)
        )
        self.classifier = nn.Sequential(
            nn.Linear(32, 1),
            nn.Sigmoid()
        )

    def forward(self, bag):
        features = self.feature_extractor(bag)
        attn_weights = self.attention(features)
        attn_weights = torch.softmax(attn_weights, dim=0)
        patient_vector = torch.sum(attn_weights * features, dim=0)
        output = self.classifier(patient_vector)
        return output, attn_weights

# 4. Trening
input_dim = bags_train_scaled[0].shape[1]
model = AttentionMIL(input_dim=input_dim)

criterion = nn.BCELoss()
# UWAGA: Zmniejszyliśmy lr z 0.001 na 0.0005 dla większej stabilności
optimizer = optim.Adam(model.parameters(), lr=0.0005)

epochs = 20
print("Rozpoczęto trening Attention-MIL (ze znormalizowanymi danymi)...")

for epoch in range(epochs):
    model.train()
    total_loss = 0
    for i in range(len(bags_train_scaled)):
        bag_tensor = torch.tensor(bags_train_scaled[i], dtype=torch.float32)
        label_tensor = torch.tensor([y_train[i]], dtype=torch.float32)

        optimizer.zero_grad()
        prediction, _ = model(bag_tensor)

        loss = criterion(prediction, label_tensor)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    if (epoch+1) % 5 == 0:
        print(f"Epoka {epoch+1}/{epochs}, Loss: {total_loss/len(bags_train_scaled):.4f}")

# 5. Ewaluacja
model.eval()
y_pred_test = []
with torch.no_grad():
    for i in range(len(bags_test_scaled)):
        bag_tensor = torch.tensor(bags_test_scaled[i], dtype=torch.float32)
        prediction, _ = model(bag_tensor)
        y_pred_test.append(1 if prediction.item() > 0.5 else 0)

bacc_mil = balanced_accuracy_score(y_test, y_pred_test)
print(f"\nWynik Balanced Accuracy (Attention-MIL): {bacc_mil * 100:.2f}%")
print("Macierz pomyłek:\n", confusion_matrix(y_test, y_pred_test))

Rozpoczęto trening Attention-MIL (ze znormalizowanymi danymi)...
Epoka 5/20, Loss: 0.6228
Epoka 10/20, Loss: 0.5914
Epoka 15/20, Loss: 0.5317
Epoka 20/20, Loss: 0.4897

Wynik Balanced Accuracy (Attention-MIL): 67.06%
Macierz pomyłek:
 [[16  1]
 [ 6  4]]
